# 02 - Retrieval Testing

The goal of this notebook is to test retrieval on its own, before adding the LLM. If the retriever returns the wrong chunk, the LLM can't give the right answer.

I load the vector database built by `src/ingest.py`, look at the relevance scores for a set of test questions, and use them to choose `K` and `SCORE_THRESHOLD`.

This notebook calls the OpenAI API to embed each question, so it needs `OPENAI_API_KEY` in `.env`.

In [16]:
# reload src/config.py automatically whenever it changes
%load_ext autoreload
%autoreload 2

# import project settings
import sys
sys.path.append("..")
from src import config

# import packages
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load the vector database

I load the database saved by `ingest.py` rather than building it again. Creating it here would add a second copy of every chunk. The embedding model must be the same one used to build the database, otherwise the question and chunk vectors can't be compared.

In [17]:
# create the embeddings
embeddings = OpenAIEmbeddings(model=config.EMBEDDING_MODEL)

# vector database
vectorstore = Chroma(
               persist_directory=str(config.VECTOR_DB_DIR),
               collection_name=config.COLLECTION_NAME,
               embedding_function=embeddings,
               )

print("Chunks in db:", len(vectorstore.get()["ids"]))

Chunks in db: 32


## One question, top 5 results

Vector search always returns `k` results, whether they are relevant or not, so I look at all five scores without a threshold.

The question mentions Easter, which isn't in the FAQ, so this checks whether retrieval finds the answer by meaning rather than by matching words.

In [18]:
query = "What hours are you open on Easter?"

results = vectorstore.similarity_search_with_relevance_scores(query, k=5)

for doc,score in results:
    question = doc.page_content.split("\n")[0]
    print(doc.metadata["id"], round(score, 3), question)

0003 0.498 Q: What are your store hours? Do they change on holidays?  
0008 0.335 Q: When are delivery windows available? Can I reschedule?  
0025 0.278 Q: Can I order special items or bulk quantities?  
0028 0.27 Q: Can I change or cancel a delivery after placing the order?  
0020 0.266 Q: Do you sell or accept gift cards?  


0003 (store hours and holidays) comes first with 0.498, well ahead of the next result at 0.335. The remaining scores bunch together around 0.27, which is what unrelated chunks score.

With the current threshold of 0.25, all five chunks would be passed to the LLM, including gift cards and bulk orders. A threshold around 0.40 would keep only 0003, but one question isn't enough to decide, so I test more questions next.

## Test questions

One question isn't enough to choose a threshold, so I wrote 15 test questions:

- **10 in-scope**, each with the FAQ id that should answer it. They are worded the way a customer might ask, not copied from the FAQ.
- **5 out-of-scope**, which the FAQ can't answer. Some are clearly unrelated (baby dolphin), and some sound like a shop question but aren't covered (gluten-free bread). The second kind is harder.

For each question I look at the top result and its score, with no threshold.

In [19]:
test_questions = [
    # in-scope: (question, expected FAQ id)
    ("What hours are you open on Easter?", "0003"),
    ("Can I return shoes I bought?", "0013"),
    ("I want to cancel my order", "0028"),
    ("Do I have to tip the driver?", "0009"),
    ("Can I pay cash for a delivery order?", "0016"),
    ("Will you match Target's prices?", "0018"),
    ("How long until my refund shows up?", "0014"),
    ("Is there a minimum spend for delivery?", "0007"),
    ("Do I need ID to buy beer?", "0022"),
    ("Are you hiring?", "0032"),

    # out-of-scope: not answered by the FAQ
    ("What is a baby dolphin called?", None),
    ("Not sure what I'm looking for.", None),
    ("Do you sell gluten-free bread?", None),
    ("Do you have a pharmacy?", None),
    ("What's the weather tomorrow?", None),
]
for question,expected_id in test_questions:

    results = vectorstore.similarity_search_with_relevance_scores(question, k=1)
    doc, score = results[0]          # k=1 gives a list with one (doc, score) pair
    top_id = doc.metadata["id"]


    if expected_id is None:
        result = "out-of-scope"
    elif top_id == expected_id:
        result = "correct"
    else:
        result = "WRONG"

    print(f"{result:<13} expected={expected_id}  top={top_id}  score={score:.3f}  {question}")

correct       expected=0003  top=0003  score=0.498  What hours are you open on Easter?
correct       expected=0013  top=0013  score=0.608  Can I return shoes I bought?
correct       expected=0028  top=0028  score=0.486  I want to cancel my order
correct       expected=0009  top=0009  score=0.661  Do I have to tip the driver?
correct       expected=0016  top=0016  score=0.541  Can I pay cash for a delivery order?
correct       expected=0018  top=0018  score=0.568  Will you match Target's prices?
correct       expected=0014  top=0014  score=0.495  How long until my refund shows up?
correct       expected=0007  top=0007  score=0.696  Is there a minimum spend for delivery?
correct       expected=0022  top=0022  score=0.527  Do I need ID to buy beer?
correct       expected=0032  top=0032  score=0.371  Are you hiring?
out-of-scope  expected=None  top=0005  score=0.090  What is a baby dolphin called?
out-of-scope  expected=None  top=0026  score=0.145  Not sure what I'm looking for.
out-of-sco

All 10 in-scope questions returned the right FAQ first.

- **Lowest correct match:** "Are you hiring?" at 0.371. It's the shortest question, so there is little meaning to match on.
- **Highest out-of-scope:** "Do you sell gluten-free bread?" at 0.299. It sounds like a shop question, so it scored much higher than "baby dolphin" (0.090).

The threshold needs to sit between 0.299 and 0.371. The original 0.25 would let two out-of-scope questions through, and 0.40 would block "Are you hiring?". I chose **0.33**, roughly the middle of the gap, and updated `config.py`.

## Check with the retriever settings

I rerun the test questions with `k=config.K` and `score_threshold=config.SCORE_THRESHOLD`, the same settings the assistant will use, and count how many chunks pass for each question.

In [20]:
for question, expected_id in test_questions:
    results = vectorstore.similarity_search_with_relevance_scores(
        question, k=config.K, score_threshold=config.SCORE_THRESHOLD
    )

    passed_ids = []
    for doc, score in results:
        passed_ids.append(doc.metadata["id"])

    if expected_id is None:
        if len(passed_ids) == 0:
            result = "blocked"      # out-of-scope and nothing got through: good
        else:
            result = "LEAKED"       # out-of-scope but chunks got through: bad
    else:
        if len(passed_ids) == 0:
            result = "MISSED"       # in-scope but nothing got through: bad
        elif passed_ids[0] == expected_id:
            result = "correct"
        else:
            result = "WRONG"

    print(f"{result:<8} expected={expected_id}  passed={len(passed_ids)} {passed_ids}  {question}")

correct  expected=0003  passed=2 ['0003', '0008']  What hours are you open on Easter?
correct  expected=0013  passed=3 ['0013', '0012', '0004']  Can I return shoes I bought?
correct  expected=0028  passed=4 ['0028', '0026', '0031', '0006']  I want to cancel my order
correct  expected=0009  passed=4 ['0009', '0007', '0029', '0015']  Do I have to tip the driver?
correct  expected=0016  passed=4 ['0016', '0015', '0009', '0007']  Can I pay cash for a delivery order?
correct  expected=0018  passed=2 ['0018', '0010']  Will you match Target's prices?
correct  expected=0014  passed=2 ['0014', '0012']  How long until my refund shows up?
correct  expected=0007  passed=4 ['0007', '0005', '0009', '0021']  Is there a minimum spend for delivery?
correct  expected=0022  passed=1 ['0022']  Do I need ID to buy beer?
correct  expected=0032  passed=1 ['0032']  Are you hiring?


No relevant docs were retrieved using the relevance score threshold 0.33


blocked  expected=None  passed=0 []  What is a baby dolphin called?


No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33


blocked  expected=None  passed=0 []  Not sure what I'm looking for.
blocked  expected=None  passed=0 []  Do you sell gluten-free bread?


No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33


blocked  expected=None  passed=0 []  Do you have a pharmacy?
blocked  expected=None  passed=0 []  What's the weather tomorrow?


The "No relevant docs were retrieved" warnings come from LangChain when the threshold removes every chunk. For the out-of-scope questions, that's the result I want.

## Summary

- With `SCORE_THRESHOLD = 0.33` and `K = 4`, **all 10 in-scope questions returned the right FAQ first**, and **all 5 out-of-scope questions returned no chunks**.
- The gap between the weakest correct match (0.371) and the strongest out-of-scope question (0.299) is small, so the threshold is tuned to these 15 questions. A new question could still fall on the wrong side, and the prompt's fallback rule is a second safety net.
- Four questions reached the `K = 4` limit. The extra chunks were related delivery FAQs rather than noise, and the right FAQ was always first, so I kept `K = 4`.
- Similarity scores aren't percentages. Even the best matches were only 0.37 to 0.70, so the threshold has to be chosen from real scores rather than guessed.

**Next:** build the RAG chain in `src/rag_chain.py` and test the full assistant in notebook 03.